# Elliptic++ Evaluation Notebook

This Colab notebook reproduces the dashboard's saved **preprocessing comparison matrix**, **classification metrics matrix**, all five **confusion matrices**, the proposed calibrated GRU model description, and seven before/after probability-error charts. It reads existing project result files; it does not train or change any model.

## One-time Google Drive setup

Create `MyDrive/EllipticPlusPlus/evaluation_inputs/` in Google Drive. Copy these files into that folder:

From `evaluation_metrics/before_preprocessing/artifacts/`:
- `gcn_test_predictions.csv`
- `graphsage_test_predictions.csv`
- `gru_test_predictions.csv`
- `logistic_regression_test_predictions.csv`
- `calibrated_risk_scorer_test_predictions.csv`

From `evaluation_metrics/results/`:
- `regression_results.csv`
- `classification_results.csv`
- `classification_results.json`

After this one-time copy, reruns read from mounted Drive without upload prompts. The saved evaluation population is 16,670 labeled transactions in timesteps 35–49; unknown class 3 is excluded from supervised metrics. The confusion-matrix artifact uses row-major order `[TN, FP, FN, TP]`, with licit as negative and illicit as positive.

> The before/after comparison is descriptive and does not prove preprocessing alone caused any change. Training and model-selection configurations can differ. GRU selection and risk-score calibration reuse validation timesteps 31–34, so calibration results may be optimistic.

## Proposed model in this project

The proposed scorer is the existing **Temporal GRU with a Platt-calibrated illicit probability**. The GRU uses 182 transaction features after saved training-fitted preprocessing, summarizes features by timestep, and predicts illicit class 1. A one-dimensional Platt transform calibrates the GRU logit. The scorer is not a separate future-risk architecture and does not forecast future transactions.

## Metric guide

| Metric | Meaning | Better result |
|---|---|---|
| RMSE | Root mean square probability error; penalizes larger errors more. | Lower |
| MSE | Mean squared probability error. | Lower |
| MAE | Average absolute probability error. | Lower |
| MAPE | Percentage error; undefined when true binary labels include zero. | Undefined |
| R² | Fit relative to predicting the mean label; negative values are valid. | Higher |
| NRMSE | RMSE divided by the binary target range (1). | Lower |
| MedAE | Median absolute probability error. | Lower |
| Bias / ME | Mean predicted probability minus true label. | Closer to zero |

These are probability-versus-binary-label measures, not transaction-amount errors.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from google.colab import drive

BEFORE_FILES = {
    "gcn": "gcn_test_predictions.csv",
    "graphsage": "graphsage_test_predictions.csv",
    "gru": "gru_test_predictions.csv",
    "logistic": "logistic_regression_test_predictions.csv",
    "risk_scorer": "calibrated_risk_scorer_test_predictions.csv",
}
AFTER_FILE = "regression_results.csv"
CLASSIFICATION_FILE = "classification_results.csv"
CLASSIFICATION_JSON_FILE = "classification_results.json"
drive.mount("/content/drive")
INPUT_DIR = Path("/content/drive/MyDrive/EllipticPlusPlus/evaluation_inputs")

required_names = set(BEFORE_FILES.values()) | {AFTER_FILE, CLASSIFICATION_FILE, CLASSIFICATION_JSON_FILE}
missing = sorted(name for name in required_names if not (INPUT_DIR / name).is_file())
if missing:
    raise FileNotFoundError(f"Copy these files to {INPUT_DIR}: {', '.join(missing)}")

MODEL_NAMES = {
    "gcn": "GCN",
    "graphsage": "GraphSAGE",
    "gru": "Temporal GRU",
    "logistic": "Logistic Regression",
    "risk_scorer": "Calibrated Risk Scorer",
}
MODEL_ORDER = list(BEFORE_FILES)
CLASSIFICATION_METRICS = ["accuracy", "precision", "recall", "specificity", "f1_score", "balanced_accuracy", "mcc", "cohens_kappa", "auc_roc", "log_loss"]
CLASSIFICATION_LABELS = {"accuracy": "Accuracy", "precision": "Precision", "recall": "Recall / Sensitivity", "specificity": "Specificity", "f1_score": "F1-score", "balanced_accuracy": "Balanced accuracy", "mcc": "Matthews correlation coefficient", "cohens_kappa": "Cohen’s kappa", "auc_roc": "ROC-AUC", "log_loss": "Log loss"}
EXPECTED_TEST_ROWS = 16670
TEST_TIMESTEPS = set(range(35, 50))


In [ ]:
# Load and validate the saved held-out prediction files.

def load_before_predictions(model_id, filename):
    path = INPUT_DIR / filename
    frame = pd.read_csv(path, dtype={"txId": "string"})
    required = {"txId", "timestep", "label", "probability"}
    missing_columns = required - set(frame.columns)
    if missing_columns:
        raise ValueError(f"{filename} is missing columns: {sorted(missing_columns)}")
    if len(frame) != EXPECTED_TEST_ROWS:
        raise ValueError(f"{filename}: expected {EXPECTED_TEST_ROWS} rows, found {len(frame)}")
    if frame["txId"].isna().any() or frame["txId"].duplicated().any():
        raise ValueError(f"{filename} has missing or duplicate transaction IDs")

    frame["timestep"] = pd.to_numeric(frame["timestep"], errors="raise").astype(int)
    frame["label"] = pd.to_numeric(frame["label"], errors="raise").astype(int)
    frame["probability"] = pd.to_numeric(frame["probability"], errors="raise")
    if not set(frame["timestep"]).issubset(TEST_TIMESTEPS):
        raise ValueError(f"{filename} contains timesteps outside 35–49")
    if not set(frame["label"].unique()).issubset({0, 1}):
        raise ValueError(f"{filename} must contain only labeled classes 0 and 1")
    if not np.isfinite(frame["probability"]).all() or not frame["probability"].between(0, 1).all():
        raise ValueError(f"{filename} contains invalid probabilities")
    return frame.sort_values("txId").reset_index(drop=True)


before_frames = {
    model_id: load_before_predictions(model_id, filename)
    for model_id, filename in BEFORE_FILES.items()
}
reference = before_frames[MODEL_ORDER[0]][["txId", "timestep", "label"]]
for model_id, frame in before_frames.items():
    if not frame[["txId", "timestep", "label"]].equals(reference):
        raise ValueError(f"{MODEL_NAMES[model_id]} prediction rows do not align to the common test population")


def calculate_probability_metrics(frame):
    """Use the dashboard's definitions on probability scores versus binary labels."""
    actual = frame["label"].to_numpy(dtype=float)
    probability = frame["probability"].to_numpy(dtype=float)
    error = probability - actual
    squared_error = error ** 2
    rmse = float(np.sqrt(np.mean(squared_error)))
    target_range = float(np.max(actual) - np.min(actual))
    total_sum_squares = float(np.sum((actual - np.mean(actual)) ** 2))
    return {
        "rmse": rmse,
        "mse": float(np.mean(squared_error)),
        "mae": float(np.mean(np.abs(error))),
        "mape": np.nan if np.any(actual == 0) else float(np.mean(np.abs(error / actual)) * 100.0),
        "r2": float(1.0 - np.sum(squared_error) / total_sum_squares) if total_sum_squares else np.nan,
        "nrmse": rmse / target_range if target_range else np.nan,
        "medae": float(np.median(np.abs(error))),
        "bias": float(np.mean(error)),
    }


before_values = {model_id: calculate_probability_metrics(frame) for model_id, frame in before_frames.items()}

# Read after-preprocessing metrics from the same saved CSV used by the dashboard.
after_frame = pd.read_csv(INPUT_DIR / AFTER_FILE)
required_after_columns = {"model", "task", "metric", "value", "status", "sample_count"}
if not required_after_columns.issubset(after_frame.columns):
    raise ValueError("regression_results.csv is missing required evaluation columns")
after_frame["model"] = after_frame["model"].astype(str).str.lower().str.strip()
after_frame["task"] = after_frame["task"].astype(str).str.lower().str.strip()
after_frame["metric"] = after_frame["metric"].astype(str).str.lower().str.strip()
after_frame = after_frame[after_frame["task"] == "regression"]

METRIC_INFO = {
    "rmse": ("RMSE", False, "Root Mean Square Error"),
    "mse": ("MSE", False, "Mean Squared Error"),
    "mae": ("MAE", False, "Mean Absolute Error"),
    "mape": ("MAPE", False, "Mean Absolute Percentage Error"),
    "r2": ("R²", True, "Coefficient of Determination"),
    "nrmse": ("NRMSE", False, "Normalized Root Mean Square Error"),
    "medae": ("MedAE", False, "Median Absolute Error"),
    "bias": ("Bias / ME", False, "Bias / Mean Error"),
}


def get_after_value(model_id, metric_id):
    selected = after_frame[(after_frame["model"] == model_id) & (after_frame["metric"] == metric_id)]
    if len(selected) != 1:
        raise ValueError(f"Expected one saved after-preprocessing row for {model_id}/{metric_id}; found {len(selected)}")
    row = selected.iloc[0]
    if int(row["sample_count"]) != EXPECTED_TEST_ROWS:
        raise ValueError(f"Unexpected after-preprocessing sample count for {model_id}/{metric_id}")
    status = str(row["status"]).lower()
    if status == "undefined":
        return np.nan
    if status != "calculated" or pd.isna(row["value"]):
        raise ValueError(f"After-preprocessing metric {model_id}/{metric_id} is {status}")
    value = float(row["value"])
    if not np.isfinite(value):
        raise ValueError(f"Non-finite after-preprocessing metric {model_id}/{metric_id}")
    return value


after_values = {
    model_id: {metric_id: get_after_value(model_id, metric_id) for metric_id in METRIC_INFO}
    for model_id in MODEL_ORDER
}

# Check that every model has exactly the expected saved regression metrics.
for model_id in MODEL_ORDER:
    available = set(after_frame.loc[after_frame["model"] == model_id, "metric"])
    expected = set(METRIC_INFO)
    if available != expected:
        raise ValueError(f"Saved after metrics for {model_id} differ from the expected set: {sorted(available)}")

print(f"Validated {EXPECTED_TEST_ROWS:,} common labeled transactions, timesteps 35–49, across all five models.")


# Load the saved classification metrics and confusion-matrix artifact.
classification_frame = pd.read_csv(INPUT_DIR / CLASSIFICATION_FILE)
classification_frame["model"] = classification_frame["model"].astype(str).str.lower()
classification_frame["metric"] = classification_frame["metric"].astype(str).str.lower()
if set(classification_frame["task"].astype(str).str.lower()) != {"classification"}:
    raise ValueError("classification_results.csv contains unexpected task rows")
if set(classification_frame["model"]) != set(MODEL_ORDER):
    raise ValueError("Classification results do not contain exactly the five expected models")
if set(classification_frame["metric"]) != set(CLASSIFICATION_METRICS) or len(classification_frame) != len(MODEL_ORDER) * len(CLASSIFICATION_METRICS):
    raise ValueError("Classification result metric/model coverage is incomplete or duplicated")
if (pd.to_numeric(classification_frame["sample_count"], errors="raise") != EXPECTED_TEST_ROWS).any():
    raise ValueError("Classification metrics have unexpected labeled test support")
if not (classification_frame["status"].astype(str).str.lower() == "calculated").all():
    raise ValueError("One or more saved classification metrics are not calculated")
classification_frame["value"] = pd.to_numeric(classification_frame["value"], errors="raise")
if not np.isfinite(classification_frame["value"]).all():
    raise ValueError("Classification metrics contain non-finite values")

with open(INPUT_DIR / CLASSIFICATION_JSON_FILE, encoding="utf-8") as f:
    classification_doc = json.load(f)
population = classification_doc.get("population", {})
if population.get("labeled_rows") != EXPECTED_TEST_ROWS or population.get("test_timesteps") != [35, 49]:
    raise ValueError("Confusion-matrix artifact has unexpected test population or timestep metadata")
confusion_matrices = {}
for result in classification_doc.get("model_results", []):
    model_id = str(result.get("model", "")).lower()
    values = result.get("confusion_matrix_licit_illicit")
    if model_id not in MODEL_ORDER or model_id in confusion_matrices or not isinstance(values, list) or len(values) != 4:
        raise ValueError(f"Invalid or duplicate confusion matrix for model {model_id!r}")
    matrix_values = np.asarray(values, dtype=np.int64).reshape(2, 2)
    if (matrix_values < 0).any() or int(matrix_values.sum()) != EXPECTED_TEST_ROWS:
        raise ValueError(f"Confusion matrix for {model_id} does not sum to labeled test support")
    confusion_matrices[model_id] = matrix_values
if set(confusion_matrices) != set(MODEL_ORDER):
    raise ValueError("Confusion matrices do not cover all five models")
print("Validated saved classification metrics and five confusion matrices on 16,670 labeled test rows (T35–49).")


In [ ]:
# Build the same long-form Before/After/Change matrix shown in the dashboard.
rows = []
for model_id in MODEL_ORDER:
    for metric_id, (label, higher_is_better, _) in METRIC_INFO.items():
        before = before_values[model_id][metric_id]
        after = after_values[model_id][metric_id]
        if np.isfinite(before) and np.isfinite(after):
            if metric_id == "bias":
                change = abs(before) - abs(after)  # Improvement means closer to zero.
            elif higher_is_better:
                change = after - before
            else:
                change = before - after
        else:
            change = np.nan
        rows.append({
            "Algorithm": MODEL_NAMES[model_id],
            "Metric": label,
            "Before Preprocessing": before,
            "After Preprocessing": after,
            "Change": change,
        })

matrix = pd.DataFrame(rows)
fmt = lambda value: "Undefined" if pd.isna(value) else f"{value:.4f}"
display(Markdown("## Preprocessing Impact Matrix\n\nChange is positive when the result improves: After − Before for R², Before − After for error metrics, and reduction in absolute bias for Bias / ME."))
display(matrix.style.format({
    "Before Preprocessing": fmt,
    "After Preprocessing": fmt,
    "Change": fmt,
}, na_rep="Undefined").set_table_styles([
    {"selector": "th", "props": [("background-color", "#eee9d9"), ("color", "#242820"), ("text-align", "left")]},
    {"selector": "td", "props": [("padding", "7px 10px"), ("border-bottom", "1px solid #ded9ca")]},
]).hide(axis="index"))

explanations = pd.DataFrame([
    ("RMSE", "Root mean square probability error; large errors are penalized more.", "Lower is better."),
    ("MSE", "Mean squared probability error.", "Lower is better."),
    ("MAE", "Average absolute probability error.", "Lower is better."),
    ("MAPE", "Percentage error cannot be defined when true labels include zero.", "Undefined; no chart."),
    ("R²", "Fit relative to predicting the mean binary label; negative values are valid.", "Higher is better."),
    ("NRMSE", "RMSE divided by the range of true labels (range = 1 for this binary test set).", "Lower is better."),
    ("MedAE", "Median absolute probability error; robust to large outliers.", "Lower is better."),
    ("Bias / ME", "Mean predicted probability minus true label; positive overpredicts, negative underpredicts.", "Closer to zero is better."),
], columns=["Metric", "Meaning", "Interpretation"])
display(Markdown("## Metric explanations"))
display(explanations)


## Classification metrics matrix

The table below is read from the saved `classification_results.csv`. Metrics use labeled test transactions only (classes 1 and 2); class 3 is excluded. Values are empirical held-out results at the saved decision protocol, not estimates recomputed by this notebook.


In [ ]:
classification_display = classification_frame.pivot(index="metric", columns="model", values="value").reindex(index=CLASSIFICATION_METRICS, columns=MODEL_ORDER)
classification_display.index = [CLASSIFICATION_LABELS[x] for x in classification_display.index]
classification_display.columns = [MODEL_NAMES[x] for x in classification_display.columns]
display(classification_display.style.format("{:.4f}").set_caption("Saved classification metrics — labeled test period T35–49 (n=16,670)").set_table_styles([
    {"selector": "th", "props": [("background-color", "#eee9d9"), ("color", "#242820"), ("text-align", "left")]},
    {"selector": "td", "props": [("padding", "7px 10px"), ("border-bottom", "1px solid #ded9ca")]},
]))


## Confusion matrices

Counts come directly from `classification_results.json`; each matrix is plotted as true class by predicted class. Rows are true licit/illicit and columns are predicted licit/illicit. Class 3 is not part of these labeled test matrices.


In [ ]:
fig, axes = plt.subplots(1, len(MODEL_ORDER), figsize=(20, 4.5))
for ax, model_id in zip(axes, MODEL_ORDER):
    cm = confusion_matrices[model_id]
    image = ax.imshow(cm, cmap="YlGn")
    ax.set_title(MODEL_NAMES[model_id], fontsize=11, weight="bold")
    ax.set_xticks([0, 1], labels=["Licit", "Illicit"])
    ax.set_yticks([0, 1], labels=["Licit", "Illicit"])
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    threshold = cm.max() / 2
    for row in range(2):
        for col in range(2):
            ax.text(col, row, f"{cm[row, col]:,}", ha="center", va="center", color="white" if cm[row, col] > threshold else "#242820", fontsize=11, weight="bold")
fig.suptitle("Saved held-out confusion matrices — class 3 excluded", fontsize=14, weight="bold")
fig.tight_layout()
plt.show()


## Proposed model: Temporal GRU with calibrated illicit probability

**Pipeline:** 182 transaction features → saved training-fitted imputation/scaling/value transform → timestep-level feature summaries → unidirectional Temporal GRU → illicit-class logit → saved one-dimensional Platt calibration → calibrated illicit probability.

This is the project's proposed scoring pipeline, not an additional independent model or a future-event predictor. The test-period classification metrics and confusion matrix above are read from the saved risk-scorer result. The calibrated risk is still an empirical score, not proof of illicit activity. Calibration and GRU selection both use validation timesteps 31–34, which can make reported calibrated performance optimistic. The held-out evaluation is timesteps 35–49, with 16,670 labeled rows; unknown class 3 is excluded from supervised metrics.


In [ ]:
risk_id = "risk_scorer"
risk_metrics = classification_frame[classification_frame["model"] == risk_id].set_index("metric")["value"]
risk_matrix = confusion_matrices[risk_id]
print("Proposed scoring pipeline: Temporal GRU + saved Platt calibration")
print(f"Test support: {EXPECTED_TEST_ROWS:,} labeled transactions (T35–49)")
print(f"Accuracy: {risk_metrics['accuracy']:.4f} | Precision: {risk_metrics['precision']:.4f} | Recall: {risk_metrics['recall']:.4f} | F1: {risk_metrics['f1_score']:.4f} | PR-AUC is shown in the saved preprocessing comparison matrix")
print(f"Confusion matrix [[TN, FP], [FN, TP]]: {risk_matrix.tolist()}")


In [ ]:
# Seven individual grouped bar charts. MAPE is intentionally excluded.
CHART_METRICS = [
    ("rmse", "Root Mean Square Error (RMSE)", "Probability error"),
    ("mse", "Mean Squared Error (MSE)", "Squared probability error"),
    ("mae", "Mean Absolute Error (MAE)", "Absolute probability error"),
    ("r2", "Coefficient of Determination (R²)", "R²"),
    ("nrmse", "Normalized Root Mean Square Error (NRMSE)", "Range-normalized probability error"),
    ("medae", "Median Absolute Error (MedAE)", "Absolute probability error"),
    ("bias", "Bias / Mean Error", "Signed mean error"),
]
BEFORE_COLOR = "#b58b32"  # gold
AFTER_COLOR = "#596b45"   # olive
x = np.arange(len(MODEL_ORDER))
bar_width = 0.36

for metric_id, title, y_label in CHART_METRICS:
    before = np.array([before_values[model][metric_id] for model in MODEL_ORDER], dtype=float)
    after = np.array([after_values[model][metric_id] for model in MODEL_ORDER], dtype=float)
    if not np.isfinite(before).all() or not np.isfinite(after).all():
        raise ValueError(f"Cannot graph missing or undefined values for {metric_id}")

    fig, ax = plt.subplots(figsize=(12, 6), facecolor="#f7f3e8")
    ax.set_facecolor("#fffefa")
    before_bars = ax.bar(x - bar_width / 2, before, bar_width, label="Before Preprocessing", color=BEFORE_COLOR)
    after_bars = ax.bar(x + bar_width / 2, after, bar_width, label="After Preprocessing", color=AFTER_COLOR)
    ax.set_title(title, fontsize=15, weight="bold", color="#242820", pad=14)
    ax.set_ylabel(y_label, color="#42463b")
    ax.set_xlabel("Algorithm", color="#42463b")
    ax.set_xticks(x, [MODEL_NAMES[model] for model in MODEL_ORDER], rotation=12, ha="right")
    ax.grid(axis="y", color="#ded9ca", linewidth=0.8, alpha=0.8)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines[["left", "bottom"]].set_color("#cfc8b8")
    ax.tick_params(colors="#56594f")
    ax.legend(frameon=False, ncols=2, loc="best")
    ax.bar_label(before_bars, fmt="%.4f", padding=3, fontsize=8)
    ax.bar_label(after_bars, fmt="%.4f", padding=3, fontsize=8)

    all_values = np.concatenate([before, after])
    if metric_id == "r2":
        # Include the actual negative range; the axis does not start at zero.
        low, high = float(all_values.min()), float(all_values.max())
        pad = max((high - low) * 0.08, 0.05)
        ax.set_ylim(low - pad, high + pad)
        ax.axhline(0, color="#a95543", linewidth=1.7, label="Zero reference")
        ax.set_title(title + " — higher is better; negative values retained", fontsize=13, weight="bold", color="#242820", pad=14)
    elif metric_id == "bias":
        # Keep the signed values and center the visible zero line in the displayed range.
        low, high = min(0.0, float(all_values.min())), max(0.0, float(all_values.max()))
        pad = max((high - low) * 0.10, 0.02)
        ax.set_ylim(low - pad, high + pad)
        ax.axhline(0, color="#a95543", linewidth=1.8, label="Zero reference")
        ax.set_title(title + " — values closer to zero indicate less systematic bias", fontsize=13, weight="bold", color="#242820", pad=14)
    else:
        ax.set_ylim(0, float(all_values.max()) * 1.20)
        ax.set_title(title + " — lower is better", fontsize=14, weight="bold", color="#242820", pad=14)

    fig.tight_layout()
    plt.show()
